In [9]:

# filepath: /Users/timwyse/cooperative_ai/analysis/md_to_latex.py
import re
from pathlib import Path

def md_to_latex_promptboxes(md_path: str, output_path: str | None = None) -> str:
    """
    Reads a negotiation transcript MD file and outputs LaTeX with promptbox environments.
    """
    text = Path(md_path).read_text()

    # Extract everything from "FULL NEGOTIATION TRANSCRIPT:" onwards
    transcript_match = re.search(r"FULL NEGOTIATION TRANSCRIPT:\s*-+\s*\n(.+)", text, re.DOTALL)
    if not transcript_match:
        raise ValueError("Could not find 'FULL NEGOTIATION TRANSCRIPT:' in the file")
    transcript = transcript_match.group(1)

    # Split into blocks by speaker headers
    # Match patterns like [PLAYER 0 (assistant)]: or [PLAYER 1 / SYSTEM (user)]: or [SYSTEM]:
    pattern = r"\[(PLAYER 0 \(assistant\)|PLAYER 1 / SYSTEM \(user\)|SYSTEM)\]:\s*"
    parts = re.split(pattern, transcript)

    # parts = [before_first_match, speaker1, content1, speaker2, content2, ...]
    # Skip the preamble (system prompt)
    turns = []
    for i in range(1, len(parts) - 1, 2):
        speaker = parts[i].strip()
        content = parts[i + 1].strip()
        turns.append((speaker, content))

    # Find the starting turn: PLAYER 0's first real message
    start_idx = None
    for i, (speaker, content) in enumerate(turns):
        if speaker == "PLAYER 0 (assistant)" and not content.startswith("(Selfish agent"):
            start_idx = i
            break

    if start_idx is None:
        raise ValueError("Could not find a starting turn from PLAYER 0 (assistant)")


    turns = turns[start_idx:]

    # Map speaker names
    speaker_map = {
        "PLAYER 0 (assistant)": "P-Red",
        "PLAYER 1 / SYSTEM (user)": "P-Blue",
        "SYSTEM": "System",
    }

    # Color map for promptbox
    color_map = {
        "P-Red": "red!10",
        "P-Blue": "blue!10",
        "System": "gray!10",
    }

    def escape_latex(s: str) -> str:
        """Escape special LaTeX characters, preserving intentional formatting."""
        # Remove "Turn: N:" prefixes
        s = re.sub(r"^Turn:?\s*\d+:\s*", "", s)
        s = re.sub(r"\n\s*Turn:?\s*\d+:\s*", "\n", s)

        # Escape special chars
        s = s.replace("\\", "\\textbackslash{}")
        s = s.replace("&", "\\&")
        s = s.replace("%", "\\%")
        s = s.replace("$", "\\$")
        s = s.replace("#", "\\#")
        s = s.replace("_", "\\_")
        s = s.replace("{", "\\{")
        s = s.replace("}", "\\}")
        s = s.replace("~", "\\textasciitilde{}")
        s = s.replace("^", "\\textasciicircum{}")

        # Fix back the textbackslash we just broke
        s = s.replace("\\textbackslash\\{\\}", "\\textbackslash{}")

        # Convert **bold** to \textbf{}
        s = re.sub(r"\*\*(.+?)\*\*", r"\\textbf{\1}", s)

        # Convert *italic* to \textit{}
        s = re.sub(r"\*(.+?)\*", r"\\textit{\1}", s)

        # Convert markdown bullet points to \item
        lines = s.split("\n")
        in_itemize = False
        new_lines = []
        for line in lines:
            stripped = line.strip()
            if stripped.startswith("- "):
                if not in_itemize:
                    new_lines.append("\\begin{itemize}")
                    in_itemize = True
                new_lines.append("  \\item " + stripped[2:])
            else:
                if in_itemize:
                    new_lines.append("\\end{itemize}")
                    in_itemize = False
                new_lines.append(line)
        if in_itemize:
            new_lines.append("\\end{itemize}")
        s = "\n".join(new_lines)

        # Blank lines → \vspace
        s = re.sub(r"\n{2,}", "\n\n\\\\vspace{0.5em}\n\n", s)

        return s

    latex_lines = []
    


    # Map speaker to box environment
    box_map = {
        "P-Red": "redbox",
        "P-Blue": "bluebox",
        "System": "graybox",
    }

    for speaker_raw, content in turns:
        display_name = speaker_map.get(speaker_raw, speaker_raw)
        box_env = box_map.get(display_name, "graybox")
        escaped = escape_latex(content)

        latex_lines.append(f"\\begin{{{box_env}}}[{display_name}]")
        latex_lines.append(escaped)
        latex_lines.append(f"\\end{{{box_env}}}")
        latex_lines.append("")

    
    result = "\n".join(latex_lines)

    if output_path:
        Path(output_path).write_text(result)
        print(f"Wrote LaTeX to {output_path}")

    return result


# --- Run it ---
latex = md_to_latex_promptboxes(
    "analysis/HAIKU_sample_contract.md",
    "analysis/HAIKU_sample_contract.tex",
)
# print(latex)


Wrote LaTeX to analysis/HAIKU_sample_contract.tex


In [10]:
md_to_latex_promptboxes(
    "analysis/GPT_4_1_sample_contract.md",
    "analysis/GPT_4_1_PPC.tex",
)

Wrote LaTeX to analysis/GPT_4_1_PPC.tex


'\\begin{redbox}[P-Red]\nLet’s analyze the situation step by step:\n\n\\vspace{0.5em}\n\n\\begin{itemize}\n  \\item I need to reach (3, 3).\n  \\item My best chip path (minimizing what I lack) is:\n\\end{itemize}\n  [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (3, 2), (3, 3)]  \n  Chips required (total): B: 2, R: 3, G: 1.  \n  Inventory: B: 0 (I’m missing 2 blue and cannot make it without help), G: 2, R: 14.\n\n\\vspace{0.5em}\n\n\\begin{itemize}\n  \\item You have 14 B, 2 G, 0 R — plenty of blue.\n\\end{itemize}\n\n\\vspace{0.5em}\n\n\\begin{itemize}\n  \\item Your likely path needs R, which you don’t have, and I have 14.\n\\end{itemize}\n\n\\vspace{0.5em}\n\nSo: The only way either of us can get to the goal is by covering the color we don’t have, using the ‘pay for other’ trading rule.\n\n\\vspace{0.5em}\n\nIf we don’t cooperate, neither of us gets to our goal. I need you for blue, you need me for red.\n\n\\vspace{0.5em}\n\nBoth of us can reach the goal easily if we cooperate, and both h